# Notebook 01 — Synthetic Dataset Synthesis

**Project:** TCO Optimisation Model — Sprint One  
**Author:** Soham Dharne (2026)  
**NFR compliance:** NFR-07 (reproducibility via fixed seed), NFR-09 (100% annotated)

---

## 1. Why Synthetic Data?

In a greenfield research context, real project records containing all 17 features defined in the IEEE SRS are unavailable or commercially sensitive. Synthetic generation offers three key advantages:

1. **Full label control** — we encode domain-expert business rules directly into the label assignment function, ensuring the dataset reflects realistic decision boundaries without requiring hundreds of hand-labelled real projects.
2. **Reproducibility (NFR-07)** — with a fixed random seed (42), the exact same dataset can be regenerated at any time, making experiments fully auditable.
3. **Class balance by design** — real TCO datasets skew heavily toward Hybrid. By controlling sampling distributions we produce a dataset with sufficient representation of all three classes for robust ML training.

The methodology follows the IEEE SRS specification approach: features are sampled from probability distributions matching domain priors (e.g., 30% Low complexity, 40% Medium, 30% High), and labels are assigned by a deterministic scoring function that mirrors TCO research findings (Dharne, 2026).

## 2. IEEE SRS Feature Schema (17 Features)

The dataset covers three feature types across 17 columns plus 2 targets:

### 2a. Numeric Features (5)
| Feature | Type | Range | Business meaning |
|---|---|---|---|
| `estimated_loc` | int | 500–200,000 | Lines of code — proxy for project scope |
| `timeline_days` | int | 7–365 | Project deadline in days |
| `team_size_required` | int | 1–20 | Headcount needed |
| `integration_count` | int | 0–20 | Number of external system integrations |
| `testing_coverage_pct` | float | 20–98 | Required test coverage percentage |

### 2b. Ordinal Features (8)
| Feature | Levels (low → high) |
|---|---|
| `complexity_score` | Low → Medium → High |
| `technical_risk_level` | Low → Medium → High → Critical |
| `seniority_required` | Junior → Mid → Senior → Architect |
| `documentation_level` | Minimal → Standard → Comprehensive |
| `performance_tier` | Standard → High → Real-time |
| `security_criticality` | Low → Medium → High → Critical |
| `budget_pressure` | Flexible → Moderate → Tight |
| `maintainability_req` | Low → Medium → High |

### 2c. Binary Feature (1)
| Feature | Values |
|---|---|
| `regulatory_compliance` | 0 (not regulated) / 1 (regulated) |

### 2d. Nominal Feature (1)
| Feature | Categories |
|---|---|
| `domain_category` | Web, Infrastructure, Security, Data, Mobile |

### 2e. Target Variables (2)
| Target | Type | Meaning |
|---|---|---|
| `target_team_label` | string (3-class) | Human / Hybrid / AI |
| `profit_margin_pct` | float | Estimated project profit margin (0–75%) |

## 3. Business-Rule Label Assignment Logic

Labels are **not** assigned randomly — they follow a priority-ordered rule tree encoding domain expertise:

### Priority 1: Human
Triggered by any **safety-critical or compliance** signal:
- Regulatory compliance = 1 AND risk in {High, Critical}
- Security criticality = Critical (unconditional)
- Security = High AND risk in {High, Critical}
- Technical risk = Critical (unconditional)
- Complexity = High AND seniority = Architect
- Performance = Real-time AND risk = Critical
- LOC > 60,000 AND complexity = High

**Rationale:** These conditions identify projects where AI hallucination risk, comprehension debt, or regulatory/security penalties make AI autonomy economically catastrophic.

### Priority 2: AI
Triggered when project is clearly **simple, safe, and budget-driven**:
- Complexity = Low AND risk ∈ {Low, Medium} AND security ∈ {Low, Medium} AND regulatory = 0 AND LOC < 10,000
- Budget = Tight AND complexity = Low AND risk = Low
- Complexity = Low AND seniority ∈ {Junior, Mid} AND risk = Low AND regulatory = 0

**Rationale:** Small, low-risk projects have AI marginal cost ~INR 100/1,000 LOC, beating human developer cost. No regulatory constraints means error recovery is inexpensive.

### Priority 3: Hybrid
Everything in between — projects where orchestrated human+AI maximises throughput without incurring full hallucination/rework risk.

A 5% label flip to the adjacent class simulates real-world judgement-call uncertainty at decision boundaries without destroying the dominant signal.

## 4. Profit Margin Formula

The profit margin is **fully derivable from input features** (independent of the team label), enabling the regression model to learn with low MAPE:

```
margin = 100% - labour_cost - risk_cost - overhead_cost - complexity_cost + budget_benefit + noise

where:
  labour_cost     = 30 + 20·seniority_norm + 10·team_norm + 5·loc_norm
  risk_cost       = 5·risk_norm + 5·security_norm + 3·regulatory + 2·integration_norm
  overhead_cost   = 4·doc_norm + 3·test_norm + 3·perf_norm + 2·maint_norm
  complexity_cost = 10·complexity_norm
  budget_benefit  = 5·budget_pressure_norm  (tight budget forces efficiency)
  noise           ~ N(0, 1.5)              (real-world variance)
  margin          clipped to [0, 75]
```

All `_norm` terms are linearly scaled to [0, 1] from their ordinal or continuous range. The formula is transparent and learnable, satisfying NFR-09 (documented derivation) and enabling MAPE ≤ 5% (NFR-08).

## 5. Environment Setup

We add `src/` to `sys.path` so that `data_synthesis`, `config`, and `utils` are importable without installing the package. This is the standard approach for repository-style ML projects.

In [ ]:
import sys
import os

# Navigate from notebooks/ up one level to project root, then into src/
NOTEBOOK_DIR = os.path.abspath('')
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f'Project root : {PROJECT_ROOT}')
print(f'src/ on path : {SRC_DIR}')

## 6. Imports

- `synthesise` — generates the synthetic DataFrame with labels and profit margins
- `save_dataset` — writes the CSV to `data/processed/` and the audit JSON to `data/synthetic/`
- `config` constants — accessed for programmatic schema display
- Standard visualisation stack: `matplotlib`, `seaborn`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

from data_synthesis import synthesise, save_dataset
from config import (
    N_SAMPLES, SEED,
    NUMERIC_FEATURES, ORDINAL_FEATURES, BINARY_FEATURES,
    NOMINAL_FEATURES, DOMAIN_CATEGORIES,
    TARGET_CLASS, TARGET_REGR, CLASS_LABELS,
    DATA_PROCESSED, DATA_SYNTHETIC,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

print(f'Config: N_SAMPLES={N_SAMPLES}, SEED={SEED}')
print(f'Output: {DATA_PROCESSED / "greenfield_features.csv"}')

## 7. Feature Schema Inspection

We print the schema programmatically from `config.py` — any future schema change propagates here automatically, keeping this notebook self-documenting (NFR-09).

In [ ]:
print('=== NUMERIC FEATURES (5) ===')
for f in NUMERIC_FEATURES:
    print(f'  {f}')

print('\n=== ORDINAL FEATURES (8) ===')
for f, levels in ORDINAL_FEATURES.items():
    print(f'  {f:28s}: {" → ".join(levels)}')

print('\n=== BINARY FEATURES (1) ===')
for f in BINARY_FEATURES:
    print(f'  {f}: 0 or 1')

print('\n=== NOMINAL FEATURES (1) ===')
print(f'  domain_category: {DOMAIN_CATEGORIES}')

print('\n=== TARGET VARIABLES ===')
print(f'  Classification : {TARGET_CLASS}  →  {CLASS_LABELS}')
print(f'  Regression     : {TARGET_REGR}  →  [0, 75] %')

total = len(NUMERIC_FEATURES) + len(ORDINAL_FEATURES) + len(BINARY_FEATURES) + len(NOMINAL_FEATURES)
print(f'\nTotal input features: {total}')

## 8. Running Synthesis

We call `synthesise(n=N_SAMPLES, seed=SEED)`. Internally this:
1. Draws 800 records using `numpy.random.default_rng(42)` — deterministic and reproducible
2. Applies the priority-ordered business rule tree to assign labels
3. Applies the cost-model formula to compute profit margins
4. Applies a 5% boundary flip to simulate real-world label uncertainty

In [ ]:
df = synthesise(n=N_SAMPLES, seed=SEED)

print(f'Shape         : {df.shape}')
print(f'Columns       : {list(df.columns)}')
print(f'Null values   : {df.isnull().sum().sum()}')
print()
df.head()

## 9. Numeric Feature Statistics

We verify generated ranges against expected distributions. The `estimated_loc` column uses a log-normal distribution (σ=0.5) centred on complexity-dependent baselines (2K / 12K / 50K LOC), clipped to [500, 200,000].

In [ ]:
df[NUMERIC_FEATURES].describe().T.style.format('{:.1f}')

## 10. Class Distribution

Expected distribution from domain knowledge:
- **Human**: ~10–20% (safety/compliance cases are relatively rare in greenfield projects)
- **Hybrid**: ~40–55% (majority of real projects fall in the middle ground)
- **AI**: ~30–45% (simple web/data/mobile projects suitable for full automation)

The actual distribution depends on how many records satisfy the Human and AI rule conditions — Human is guarded by conservative safety rules, AI by strict simplicity criteria.

In [ ]:
label_counts = df[TARGET_CLASS].value_counts()
label_pct    = df[TARGET_CLASS].value_counts(normalize=True) * 100

print('=== Class Distribution ===')
for label in CLASS_LABELS:
    n = label_counts.get(label, 0)
    p = label_pct.get(label, 0.0)
    print(f'  {label:8s}: {n:4d} records  ({p:.1f}%)')

colors = {'Human': '#E53935', 'Hybrid': '#FB8C00', 'AI': '#43A047'}

fig, ax = plt.subplots(figsize=(7, 4))
vals = [label_counts.get(l, 0) for l in CLASS_LABELS]
bars = ax.bar(CLASS_LABELS, vals, color=[colors[l] for l in CLASS_LABELS],
               width=0.5, edgecolor='white', linewidth=1.5)
for bar, lbl in zip(bars, CLASS_LABELS):
    n = label_counts.get(lbl, 0)
    p = label_pct.get(lbl, 0.0)
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 4,
            f'{n}\n({p:.1f}%)',
            ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Record count', fontsize=11)
ax.set_title('Synthesised Dataset — Class Distribution (N=800)', fontsize=12)
plt.tight_layout()
plt.show()

## 11. Profit Margin Distribution

We inspect the regression target. The distribution should be roughly bell-shaped within [0, 75]%, with:
- **Lower margins** for Human-label records (high seniority + complexity = high labour cost)
- **Higher margins** for AI-label records (low complexity, often tight-budget efficiency gains)
- **Hybrid** falling in between

This separation by label class validates that the regression target carries meaningful signal for the downstream regression model.

In [ ]:
print('=== Profit Margin Stats ===')
print(df[TARGET_REGR].describe())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: overall histogram
axes[0].hist(df[TARGET_REGR], bins=40, color='#2196F3', edgecolor='white', alpha=0.85)
axes[0].axvline(df[TARGET_REGR].mean(), color='red', linestyle='--',
                label=f'Mean = {df[TARGET_REGR].mean():.1f}%')
axes[0].set_xlabel('Profit Margin (%)', fontsize=11)
axes[0].set_ylabel('Count')
axes[0].set_title('Overall Profit Margin Distribution')
axes[0].legend()

# Right: per-class KDE
for label, color in [('Human', '#E53935'), ('Hybrid', '#FB8C00'), ('AI', '#43A047')]:
    subset = df[df[TARGET_CLASS] == label][TARGET_REGR]
    axes[1].hist(subset, bins=25, alpha=0.55, label=label, color=color, edgecolor='white')
axes[1].set_xlabel('Profit Margin (%)', fontsize=11)
axes[1].set_ylabel('Count')
axes[1].set_title('Profit Margin by Label Class')
axes[1].legend(title='Team label')

plt.tight_layout()
plt.show()

## 12. Ordinal Feature Distributions

Each ordinal feature is plotted in level order to verify the sampling probabilities from `_make_record`. For example:
- `complexity_score`: expected ≈ 30% Low / 40% Medium / 30% High
- `technical_risk_level`: expected ≈ 25% Low / 35% Medium / 30% High / 10% Critical
- `budget_pressure`: expected ≈ 30% Flexible / 40% Moderate / 30% Tight

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, (col, levels) in enumerate(ORDINAL_FEATURES.items()):
    counts = df[col].value_counts().reindex(levels, fill_value=0)
    axes[i].bar(levels, counts.values, color='#5C6BC0', alpha=0.85, edgecolor='white')
    axes[i].set_title(col, fontsize=9)
    axes[i].set_ylabel('Count', fontsize=8)
    axes[i].tick_params(axis='x', labelrotation=25, labelsize=8)
    for j, v in enumerate(counts.values):
        axes[i].text(j, v + 2, str(v), ha='center', fontsize=8)

plt.suptitle('Ordinal Feature Distributions (N=800)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 13. Domain Category and Regulatory Flag Distributions

The nominal `domain_category` is sampled with domain-specific probabilities: Web(30%), Data(25%), Infrastructure(20%), Security(15%), Mobile(10%).

The `regulatory_compliance` flag is not uniformly sampled — it is conditioned on domain and risk:
- Security/Data domains have +30% regulatory probability
- High/Critical risk has +20% regulatory probability

This conditioning reflects reality: financial, healthcare, and data platforms disproportionately face regulatory requirements.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Domain category
domain_counts = df['domain_category'].value_counts().reindex(DOMAIN_CATEGORIES, fill_value=0)
axes[0].bar(DOMAIN_CATEGORIES, domain_counts.values, color='#00ACC1', edgecolor='white', alpha=0.85)
axes[0].set_title('Domain Category Distribution')
axes[0].set_ylabel('Count')
for j, v in enumerate(domain_counts.values):
    axes[0].text(j, v + 2, str(v), ha='center', fontsize=9)

# Regulatory compliance breakdown by domain
reg_by_domain = df.groupby('domain_category')['regulatory_compliance'].mean() * 100
reg_by_domain = reg_by_domain.reindex(DOMAIN_CATEGORIES)
axes[1].bar(DOMAIN_CATEGORIES, reg_by_domain.values, color='#EF5350', edgecolor='white', alpha=0.85)
axes[1].set_title('Regulatory Compliance Rate by Domain (%)')
axes[1].set_ylabel('Regulated (%)')
for j, v in enumerate(reg_by_domain.values):
    axes[1].text(j, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f'Overall regulatory rate: {df["regulatory_compliance"].mean()*100:.1f}%')

## 14. Saving the Dataset

`save_dataset` performs two writes:
1. **CSV** → `data/processed/greenfield_features.csv` — canonical input for all downstream notebooks
2. **JSON audit log** → `data/synthetic/synthesis_log.json` — records generation timestamp, seed, class distribution, and margin statistics

The dual-write satisfies **NFR-09** (full audit trail) and **NFR-07** (reproducibility documentation). The `log=True` parameter is the default and should always be kept enabled in production runs.

In [ ]:
out_path = save_dataset(df, log=True)
print(f'CSV saved  → {out_path}')

log_path = DATA_SYNTHETIC / 'synthesis_log.json'
with open(log_path) as f:
    audit = json.load(f)

print('\n=== Synthesis Audit Log ===')
print(json.dumps(audit, indent=2))

## 15. Round-Trip Verification

We reload the saved CSV and verify shape, column order, and numeric precision. This confirms `save_dataset` serialises all dtypes correctly — especially float rounding for `profit_margin_pct` and string representation for the ordinal columns.

In [ ]:
df_reloaded = pd.read_csv(out_path)

assert df_reloaded.shape == df.shape, f'Shape mismatch: {df_reloaded.shape} vs {df.shape}'
assert list(df_reloaded.columns) == list(df.columns), 'Column order changed after save/reload!'

for col in NUMERIC_FEATURES + [TARGET_REGR]:
    max_diff = (df_reloaded[col] - df[col]).abs().max()
    assert max_diff < 0.01, f'Numeric drift in {col}: max_diff={max_diff:.6f}'

label_match = (df_reloaded[TARGET_CLASS] == df[TARGET_CLASS]).all()
assert label_match, 'Label column mismatch after save/reload!'

print('Round-trip verification PASSED')
print(f'Shape : {df_reloaded.shape}')
print(f'Nulls : {df_reloaded.isnull().sum().sum()}')
df_reloaded.dtypes

## 16. Summary

| Step | Outcome |
|---|---|
| Feature schema | 17 features across 4 types (5 numeric, 8 ordinal, 1 binary, 1 nominal) |
| Synthesis method | Rule-based sampling with domain priors; fixed seed=42 |
| Label assignment | Priority-ordered business rules (Human > AI > Hybrid) with 5% boundary noise |
| Profit margin | Fully feature-derivable cost model + Gaussian noise clipped to [0, 75]% |
| Output CSV | `data/processed/greenfield_features.csv` (800 records, 0 nulls) |
| Audit log | `data/synthetic/synthesis_log.json` with generation metadata |
| NFR-07 | SATISFIED — seed=42 guarantees exact reproducibility |
| NFR-09 | SATISFIED — all decisions annotated in markdown cells |

**Next step:** Notebook 02 — Exploratory Data Analysis (`02_eda.ipynb`)